# 09 - Enhanced Trajectory Simulations V2

This notebook builds upon the previous trajectory simulations with enhanced visualization techniques and advanced analysis methods.

**Converted from:** `trajectories_simulation_v2.nb` (Mathematica)

## Contents:
1. Enhanced trajectory visualization with density plots
2. Velocity and acceleration analysis
3. Energy landscape exploration
4. Long-time behavior and equilibrium
5. Escape time statistics

## Background

This notebook extends trajectory analysis with more sophisticated visualization and analytical techniques to understand SGD dynamics at a deeper level.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
from scipy.ndimage import gaussian_filter
from matplotlib.patches import Circle

# Add utils to path
sys.path.insert(0, str(Path.cwd() / 'utils'))

from sgd_simulator import SGDSimulator, SGDConfig
from loss_functions import SmoothNonlinearLoss, generate_noisy_data
from visualization import plot_loss_landscape

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✓ Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")

## 1. Setup and Data Generation

In [ ]:
# Generate data
np.random.seed(42)

x_data, y_data = generate_noisy_data(
    x_range=(-3, 3),
    n_points=20,
    n_samples_per_point=5,
    noise_std=0.5,
    p=1.0,
    random_state=42
)

loss_obj = SmoothNonlinearLoss(p=1.0)

print(f"Generated {len(x_data)} data points")

## 2. Enhanced Trajectory Visualization with Density

We'll visualize trajectories with density heatmaps showing where SGD spends most time.

In [ ]:
# Run long trajectory to accumulate statistics
config = SGDConfig(
    learning_rate=0.05,
    batch_size=10,
    n_iterations=50000,
    random_state=42
)

simulator = SGDSimulator(config)
initial_params = np.array([0.5, 2.0])

trajectory, iterations = simulator.run_trajectory(
    initial_params=initial_params,
    gradient_fn=loss_obj.gradient,
    x_data=x_data,
    y_data=y_data,
    save_every=10
)

print(f"Simulated trajectory with {len(trajectory)} saved points")
print(f"Total iterations: {config.n_iterations}")

In [ ]:
# Create density heatmap
param_range = ((-1, 3), (-1, 3))

# 2D histogram
hist, xedges, yedges = np.histogram2d(
    trajectory[:, 0], trajectory[:, 1],
    bins=80,
    range=[[param_range[0][0], param_range[0][1]],
           [param_range[1][0], param_range[1][1]]]
)

# Smooth the density
density = gaussian_filter(hist.T, sigma=2.0)

# Normalize to probability density
density = density / np.sum(density)

print("Density heatmap computed")

In [ ]:
# Plot density heatmap with loss contours
fig, ax = plt.subplots(figsize=(14, 11))

# Density heatmap
X, Y = np.meshgrid(xedges[:-1], yedges[:-1])
im = ax.imshow(density, extent=[param_range[0][0], param_range[0][1],
                                param_range[1][0], param_range[1][1]],
              origin='lower', cmap='hot', aspect='auto', alpha=0.8)
plt.colorbar(im, ax=ax, label='Time Density (Probability)')

# Overlay loss contours
n_contour = 100
a_vals = np.linspace(param_range[0][0], param_range[0][1], n_contour)
b_vals = np.linspace(param_range[1][0], param_range[1][1], n_contour)
A, B = np.meshgrid(a_vals, b_vals)

L = np.zeros_like(A)
for i in range(n_contour):
    for j in range(n_contour):
        params = np.array([A[i, j], B[i, j]])
        L[i, j] = loss_obj(params, x_data, y_data)

ax.contour(A, B, L, levels=20, colors='cyan', alpha=0.5, linewidths=1.5)

ax.set_xlabel('Parameter a', fontsize=12)
ax.set_ylabel('Parameter b', fontsize=12)
ax.set_title('SGD Trajectory Density Heatmap with Loss Contours', fontsize=14)
ax.grid(True, alpha=0.3, color='white')
plt.tight_layout()
plt.show()

print("Hot colors indicate regions where SGD spends more time")

## 3. Velocity and Acceleration Analysis

Analyze the instantaneous velocity and acceleration of SGD to understand its dynamics.

In [ ]:
# Compute velocity (finite differences)
dt = 10  # iterations between saved points
velocity = np.diff(trajectory, axis=0) / dt

# Compute acceleration
acceleration = np.diff(velocity, axis=0) / dt

# Compute magnitudes
speed = np.linalg.norm(velocity, axis=1)
accel_mag = np.linalg.norm(acceleration, axis=1)

print(f"Computed velocity for {len(velocity)} trajectory segments")
print(f"Mean speed: {np.mean(speed):.4f}")
print(f"Max speed: {np.max(speed):.4f}")

In [ ]:
# Plot velocity and acceleration over time
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Speed over time
axes[0].plot(iterations[1:], speed, 'b-', linewidth=1.5, alpha=0.7)
axes[0].axhline(np.mean(speed), color='red', linestyle='--', 
               linewidth=2, label='Mean speed')
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('Speed', fontsize=12)
axes[0].set_title('SGD Speed Over Time', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Acceleration magnitude over time
axes[1].plot(iterations[2:], accel_mag, 'g-', linewidth=1.5, alpha=0.7)
axes[1].axhline(np.mean(accel_mag), color='red', linestyle='--',
               linewidth=2, label='Mean acceleration')
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('Acceleration Magnitude', fontsize=12)
axes[1].set_title('SGD Acceleration Over Time', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Energy Landscape Exploration

Track the loss (energy) evolution and analyze oscillations around minima.

In [ ]:
# Compute loss trajectory
losses = np.array([loss_obj(params, x_data, y_data) for params in trajectory])

print(f"Loss range: [{np.min(losses):.4f}, {np.max(losses):.4f}]")
print(f"Final 1000 iterations - Mean loss: {np.mean(losses[-100:]):.4f}")
print(f"Final 1000 iterations - Std loss: {np.std(losses[-100:]):.4f}")

In [ ]:
# Plot loss evolution with moving average
fig, ax = plt.subplots(figsize=(14, 7))

# Raw loss
ax.plot(iterations, losses, 'b-', linewidth=0.5, alpha=0.3, label='Loss')

# Moving average
window_size = 50
moving_avg = np.convolve(losses, np.ones(window_size)/window_size, mode='valid')
ax.plot(iterations[window_size-1:], moving_avg, 'r-', 
       linewidth=2.5, label=f'Moving avg (window={window_size})')

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Loss Evolution with Moving Average', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze oscillations in late stage
late_stage_losses = losses[-1000:]
late_stage_iters = iterations[-1000:]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Time series of late-stage losses
axes[0].plot(late_stage_iters, late_stage_losses, 'b-', linewidth=1.5)
axes[0].axhline(np.mean(late_stage_losses), color='red', 
               linestyle='--', linewidth=2, label='Mean')
axes[0].fill_between(late_stage_iters,
                     np.mean(late_stage_losses) - np.std(late_stage_losses),
                     np.mean(late_stage_losses) + np.std(late_stage_losses),
                     color='red', alpha=0.2, label='±1 std')
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Late-Stage Loss Oscillations', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Histogram of late-stage losses
axes[1].hist(late_stage_losses, bins=30, edgecolor='black', alpha=0.7)
axes[1].axvline(np.mean(late_stage_losses), color='red', 
               linestyle='--', linewidth=2, label='Mean')
axes[1].set_xlabel('Loss', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Late-Stage Losses', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Escape Time Statistics

Analyze how long SGD takes to escape from different regions of parameter space.

In [ ]:
def compute_escape_times(trajectory, centers, radius=0.3):
    """
    Compute time spent in balls around specified centers.
    """
    escape_times = []
    
    for center in centers:
        # Find when trajectory is inside the ball
        distances = np.linalg.norm(trajectory - center, axis=1)
        inside = distances < radius
        
        # Count consecutive stays
        stays = []
        current_stay = 0
        
        for is_inside in inside:
            if is_inside:
                current_stay += 1
            else:
                if current_stay > 0:
                    stays.append(current_stay)
                current_stay = 0
        
        if current_stay > 0:
            stays.append(current_stay)
        
        escape_times.append(stays)
    
    return escape_times

# Define regions of interest
centers = [
    np.array([1.0, 1.0]),  # Center region
    np.array([2.0, 0.5]),  # Another region
    np.array([0.3, 2.2]),  # Third region
]

escape_times = compute_escape_times(trajectory, centers, radius=0.3)

for i, (center, times) in enumerate(zip(centers, escape_times)):
    if len(times) > 0:
        print(f"Region {i+1} at {center}: {len(times)} visits, "
              f"mean stay = {np.mean(times)*10:.0f} iterations")
    else:
        print(f"Region {i+1} at {center}: Never visited")

In [ ]:
# Visualize regions on loss landscape
fig, ax = plt.subplots(figsize=(14, 11))

# Loss landscape
plot_loss_landscape(
    loss_fn=loss_obj,
    x_data=x_data,
    y_data=y_data,
    param_range=param_range,
    n_points=100,
    contour_levels=30,
    ax=ax
)

# Plot trajectory (thinned for clarity)
skip = 20
ax.plot(trajectory[::skip, 0], trajectory[::skip, 1], 
       'w-', linewidth=1, alpha=0.3)

# Plot regions
colors_regions = ['red', 'orange', 'yellow']
for i, (center, times) in enumerate(zip(centers, escape_times)):
    circle = Circle(center, 0.3, color=colors_regions[i], 
                   fill=False, linewidth=3, linestyle='--')
    ax.add_patch(circle)
    
    visits = len(times)
    mean_stay = np.mean(times)*10 if len(times) > 0 else 0
    ax.plot(center[0], center[1], 'o', color=colors_regions[i], 
           markersize=15, markeredgecolor='black', markeredgewidth=2,
           label=f'Region {i+1}: {visits} visits, {mean_stay:.0f} iter avg')

ax.set_title('Escape Time Analysis - Regions of Interest', fontsize=14)
ax.legend(fontsize=11, loc='upper right')
plt.tight_layout()
plt.show()

## Summary

In this enhanced trajectory analysis, we explored:

1. **Density Visualization**: Created heatmaps showing where SGD spends most time
2. **Velocity and Acceleration**: Analyzed instantaneous dynamics of parameter updates
3. **Energy Landscape**: Tracked loss evolution and equilibrium oscillations
4. **Escape Time Statistics**: Quantified how long SGD stays in different regions

**Key Insights:**
- SGD concentrates probability density in low-loss regions (stationary distribution)
- Speed and acceleration vary significantly during optimization
- Late-stage oscillations around minima reflect the noise-induced exploration
- Different regions have different "stickiness" - some trap SGD longer than others
- Escape times provide insights into barrier heights and local geometry

**Next Steps:**
- Extend analysis to multi-dimensional examples (2D and 3D)
- Explore piecewise linear loss functions